In [53]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# ── Load all stages ───────────────────────────────────────────────────────────
df_all = pd.read_csv('sweep_results.csv')

# Stage 1: rows 34-133 (0-indexed 33:133); Stage 2: rows 134+ (0-indexed 133:)
s1 = df_all.iloc[33:133].copy()
s2 = df_all.iloc[133:].copy()
s3 = pd.read_csv('sweep_results_s3.csv')
s4 = pd.read_csv('sweep_results_s4.csv')
s5 = pd.read_csv('sweep_results_s5.csv')

# Add stage label
for df, label in [(s1,'Stage 1'),(s2,'Stage 2'),(s3,'Stage 3'),(s4,'Stage 4'),(s5,'Stage 5')]:
    df['stage_label'] = label

# Filter out crashed runs (city_rmse=999)
for name, df in [('S1',s1),('S2',s2),('S3',s3),('S4',s4),('S5',s5)]:
    n_before = len(df)
    df.drop(df[df['city_rmse'] >= 999].index, inplace=True)
    print(f'{name}: {len(df)} valid runs (dropped {n_before - len(df)} crashed)')

# Numeric columns
for df in [s1, s2, s3, s4, s5]:
    for col in ['lr','weight_decay','dropout','city_rmse','train_city_rmse','global_rmse',
                'global_mae','train_city_rmse','train_city_rmse','pred_max',
                'patch_size','batch_size','n_layers','base_channels']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    if 'oversample_factor' in df.columns:
        df['oversample_factor'] = pd.to_numeric(df['oversample_factor'], errors='coerce')

print('\nLoaded successfully.')

S1: 100 valid runs (dropped 0 crashed)
S2: 93 valid runs (dropped 0 crashed)
S3: 50 valid runs (dropped 0 crashed)
S4: 100 valid runs (dropped 0 crashed)
S5: 96 valid runs (dropped 0 crashed)

Loaded successfully.


In [61]:
import wandb
import pandas as pd

PROJECT = 'mjm060323-university-of-amsterdam/cisi-hyperparameter-sweep'
api = wandb.Api(timeout=60)

def get_best(run):
    for key in ['val_rmse.min', 'val_city_rmse.min', 'val_rmse', 'val_city_rmse']:
        v = run.summary.get(key)
        if v is None:
            continue
        if isinstance(v, (int, float)):
            return float(v)
        # SummarySubDict — try .get or direct access
        try:
            return float(v)
        except Exception:
            pass
    return None

sweeps = api.project(PROJECT.split('/')[1], entity=PROJECT.split('/')[0]).sweeps()

rows = []
for sw in sweeps:
    runs = list(sw.runs)
    best = None
    metric = sw.config.get('metric', {}).get('name', '?')
    params = list(sw.config.get('parameters', {}).keys())
    for r in runs:
        v = get_best(r)
        if v is not None:
            best = min(best, v) if best is not None else v
    created = str(sw._attrs.get('createdAt', '?'))[:10]
    rows.append({
        'sweep_id': sw.id,
        'created':  created,
        'n_runs':   len(runs),
        'metric':   metric,
        'best':     round(best, 4) if best is not None else None,
        'params':   ', '.join(sorted(params)),
    })

rows.sort(key=lambda x: x['created'])
df_sweeps = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 50)
print(df_sweeps.to_string(index=False))


sweep_id created  n_runs        metric   best                                                                                                                             params
7qoompyp       ?       0 val_city_rmse    NaN                    alpha, base_channels, batch_size, delta, dropout, kappa, loss_fn, lr, n_layers, patch_size, scale, weight_decay
v1fzevwa       ?       0 val_city_rmse    NaN                    alpha, base_channels, batch_size, delta, dropout, kappa, loss_fn, lr, n_layers, patch_size, scale, weight_decay
pwmhvsyr       ?       0 val_city_rmse    NaN                    alpha, base_channels, batch_size, delta, dropout, kappa, loss_fn, lr, n_layers, patch_size, scale, weight_decay
zaa0u92x       ?      50 val_city_rmse 0.0312                    alpha, base_channels, batch_size, delta, dropout, kappa, loss_fn, lr, n_layers, patch_size, scale, weight_decay
pfsuy6jy       ?      35 val_city_rmse 0.0315                    alpha, base_channels, batch_size, delta, dropout, 